# Jani — entrenamiento del clasificador de hojas de café

**Cómo usarlo**
1. En Colab: `Archivo → Subir notebook` y elige este archivo.
2. `Entorno de ejecución → Cambiar tipo de entorno → GPU T4`.
3. En la celda **Configuración**, elige `MODEL_NAME`.
4. `Entorno de ejecución → Ejecutar todo`.
5. Al final se descarga `jani_model_<nombre>.zip`. Descomprímelo en `app/public/models/` del repositorio.

Para entrenar otro modelo, cambia `MODEL_NAME` y vuelve a ejecutar todo. Cada modelo queda en su propia carpeta.

**Qué produce:** `model_fp32.onnx`, `model_int8.onnx`, `model_card.json` (clases, normalización, temperatura, umbrales, métricas) y `confusion_*.png`.

In [ ]:
!pip -q install timm datasets onnx onnxruntime onnxscript scikit-learn
!apt-get -qq install -y unrar > /dev/null 2>&1 || true

## Configuración

In [ ]:
# --- Elige un modelo por ejecución ---
MODEL_NAME = "efficientnet_lite0"      # principal
# MODEL_NAME = "mobilenetv3_small_100" # plan B (se usa sin cuantizar)
# MODEL_NAME = "mobilenetv2_100"       # alternativa que cuantiza bien

IMG_SIZE = 224
EPOCHS = 5
BATCH = 64
LR = 1e-3
MAX_PER_CLASS = 4000     # submuestreo de JMuBEN para entrenar rápido; None = todo
USE_BRACOL = True        # agrega hojas de Brasil si la descarga funciona
SEED = 42

# Orden canónico de clases. La app depende de este orden.
CLASSES = ["sana", "roya", "minador", "phoma", "cercospora"]

# Reglas iniciales de "no estoy segura" (se ajustan abajo con validación)
CONF_THRESHOLD = 0.70
MARGIN_THRESHOLD = 0.15

OUT_DIR = f"jani_model_{MODEL_NAME}"

In [ ]:
import os, re, io, json, glob, random, zipfile, shutil, subprocess, urllib.request, time
import numpy as np, torch, torch.nn as nn, timm
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", device)
assert device == "cuda", "Activa la GPU: Entorno de ejecución → Cambiar tipo de entorno → T4"

## 1. Datos de entrenamiento: JMuBEN (Kenia)

Se descarga del espejo en Hugging Face (`Project-AgML/arabica_coffee_leaf_disease_classification`, licencia CC BY 4.0, 58.549 imágenes, unos 1,8 GB).

In [ ]:
from datasets import load_dataset
ds = load_dataset("Project-AgML/arabica_coffee_leaf_disease_classification", split="train")
hf_names = ds.features["label"].names
print("Clases en el dataset:", hf_names, "| total:", len(ds))

def to_canonical(name):
    n = name.lower()
    if "cer" in n and "spora" in n: return "cercospora"   # cubre Cerscospora / Cescospora
    if "phoma" in n or "brown" in n: return "phoma"
    if "rust" in n or "roya" in n or n.strip() == "clr": return "roya"
    if "miner" in n or "minador" in n: return "minador"
    if "health" in n or "sana" in n or "sano" in n: return "sana"
    return None

hf_to_idx = {i: CLASSES.index(to_canonical(n)) for i, n in enumerate(hf_names)}
print("Mapeo:", {hf_names[i]: CLASSES[j] for i, j in hf_to_idx.items()})

labels_all = np.array(ds["label"])
idx_by_class = {c: np.where(labels_all == c)[0] for c in range(len(hf_names))}
train_idx, val_idx = [], []
rng = np.random.default_rng(SEED)
for c, idx in idx_by_class.items():
    idx = rng.permutation(idx)
    if MAX_PER_CLASS: idx = idx[:MAX_PER_CLASS]
    n_val = int(0.15 * len(idx))
    val_idx += idx[:n_val].tolist(); train_idx += idx[n_val:].tolist()
print("Entrenamiento:", len(train_idx), "| Validación:", len(val_idx))

## 2. Datos externos desde Mendeley Data

- **BRACOL** (Brasil): se suma al entrenamiento si `USE_BRACOL = True`.
- **Uganda** y **Perú**: solo para la **prueba fuera de distribución**. El modelo nunca los ve al entrenar.

Si una descarga automática falla, abre la página del dataset, pulsa *Download All* y sube el `.zip` a Colab con el nombre que indica el mensaje.

In [ ]:
MENDELEY = {
    "bracol": ("yy2k5y8mxg", 1, "https://data.mendeley.com/datasets/yy2k5y8mxg/1"),
    "uganda": ("k36wnd6knb", 1, "https://data.mendeley.com/datasets/k36wnd6knb/1"),
    "peru":   ("mfpxg4y65r", 1, "https://data.mendeley.com/datasets/mfpxg4y65r/1"),
}
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def extract_nested(folder):
    for _ in range(3):
        found = False
        for p in glob.glob(os.path.join(folder, "**", "*"), recursive=True):
            low = p.lower()
            try:
                if low.endswith(".zip"):
                    zipfile.ZipFile(p).extractall(os.path.splitext(p)[0]); os.remove(p); found = True
                elif low.endswith(".rar"):
                    d = os.path.splitext(p)[0]; os.makedirs(d, exist_ok=True)
                    subprocess.run(["unrar", "x", "-o+", "-inul", p, d + "/"]); os.remove(p); found = True
            except Exception as e:
                print("  No se pudo extraer", p, e)
        if not found: break

def fetch_mendeley(key):
    did, ver, page = MENDELEY[key]
    dest = f"data/{key}"
    if os.path.isdir(dest) and any(f.lower().endswith(IMG_EXT) for _,_,fs in os.walk(dest) for f in fs):
        return dest
    os.makedirs(dest, exist_ok=True)
    local_zip = f"{key}.zip"
    if not os.path.exists(local_zip):
        url = f"https://prod-dcd-datasets-cache-zipfiles.s3.eu-west-1.amazonaws.com/{did}-{ver}.zip"
        try:
            print(f"Descargando {key} ...")
            urllib.request.urlretrieve(url, local_zip)
        except Exception as e:
            print(f"  Descarga automática falló ({e}).")
            print(f"  Abre {page} → Download All → sube el archivo a Colab como '{local_zip}' y vuelve a ejecutar.")
            return None
    try:
        zipfile.ZipFile(local_zip).extractall(dest)
    except Exception as e:
        print("  Zip inválido:", e); return None
    extract_nested(dest)
    return dest

def scan_folder(root):
    # Devuelve (ruta, clase_canónica o 'desconocida') usando los nombres de las carpetas.
    items = []
    for dp, _, fs in os.walk(root):
        rel = os.path.relpath(dp, root)
        label = None
        for part in reversed(rel.split(os.sep)):
            label = to_canonical(part)
            if label: break
        for f in fs:
            if f.lower().endswith(IMG_EXT):
                items.append((os.path.join(dp, f), label or "desconocida"))
    return items

def summarize(name, items):
    from collections import Counter
    print(f"{name}: {len(items)} imágenes →", dict(Counter(l for _, l in items)))

ext = {}
for key in MENDELEY:
    if key == "bracol" and not USE_BRACOL: continue
    d = fetch_mendeley(key)
    ext[key] = scan_folder(d) if d else []
    summarize(key, ext[key])
print("\nRevisa los conteos. Si una clase conocida aparece como 'desconocida', ajusta to_canonical().")

## 3. Datasets y aumentación

In [ ]:
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=len(CLASSES)).to(device)
cfg = timm.data.resolve_data_config({}, model=model)
MEAN, STD = list(cfg["mean"]), list(cfg["std"])
print("Normalización:", MEAN, STD)

# Aumentación fuerte para acercarse a fotos de campo
train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(25),
    T.ColorJitter(0.3, 0.3, 0.3, 0.05),
    T.RandomApply([T.GaussianBlur(5)], p=0.2),
    T.ToTensor(), T.Normalize(MEAN, STD),
])
# Debe coincidir con el preprocesamiento de la app: redimensionar a 224x224 y normalizar
eval_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)])

class HFSet(Dataset):
    def __init__(self, idx, tf): self.idx, self.tf = idx, tf
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        r = ds[int(self.idx[i])]
        return self.tf(r["image"].convert("RGB")), hf_to_idx[r["label"]]

class FileSet(Dataset):
    def __init__(self, items, tf): self.items, self.tf = items, tf
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p, l = self.items[i]
        return self.tf(Image.open(p).convert("RGB")), CLASSES.index(l)

train_sets = [HFSet(train_idx, train_tf)]
bracol_known = [(p, l) for p, l in ext.get("bracol", []) if l in CLASSES]
if bracol_known:
    train_sets.append(FileSet(bracol_known, train_tf)); print("BRACOL agregado:", len(bracol_known))
train_ds = torch.utils.data.ConcatDataset(train_sets)
val_ds = HFSet(val_idx, eval_tf)
train_dl = DataLoader(train_ds, BATCH, shuffle=True, num_workers=2, drop_last=True)
val_dl = DataLoader(val_ds, BATCH, num_workers=2)

## 4. Entrenamiento

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(train_dl))
crit = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = torch.amp.GradScaler("cuda")

@torch.no_grad()
def get_logits(m, dl):
    m.eval(); L, Y = [], []
    for x, y in dl:
        L.append(m(x.to(device)).float().cpu()); Y.append(y)
    return torch.cat(L), torch.cat(Y)

best = 0
for ep in range(EPOCHS):
    model.train(); t0 = time.time(); tot = 0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        opt.zero_grad(set_to_none=True)
        with torch.autocast("cuda"):
            loss = crit(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item()
    lg, yy = get_logits(model, val_dl)
    acc = (lg.argmax(1) == yy).float().mean().item()
    print(f"Época {ep+1}/{EPOCHS} | pérdida {tot/len(train_dl):.3f} | validación {acc:.4f} | {time.time()-t0:.0f}s")
    if acc >= best:
        best = acc; torch.save(model.state_dict(), f"{OUT_DIR}/best.pt")
model.load_state_dict(torch.load(f"{OUT_DIR}/best.pt")); model.eval()
print("Mejor precisión de validación:", round(best, 4))
print("OJO: JMuBEN trae aumentación, así que este número está inflado. El número honesto es el de la sección 6.")

## 5. Calibración de la confianza (temperature scaling)

In [ ]:
val_logits, val_y = get_logits(model, val_dl)
logT = torch.zeros(1, requires_grad=True)
o = torch.optim.LBFGS([logT], lr=0.1, max_iter=100)
def _closure():
    o.zero_grad(); l = nn.functional.cross_entropy(val_logits / logT.exp(), val_y); l.backward(); return l
o.step(_closure)
TEMPERATURE = float(logT.exp())
print("Temperatura:", round(TEMPERATURE, 3))

def decide(logits, T=None, conf=CONF_THRESHOLD, margin=MARGIN_THRESHOLD):
    p = torch.softmax(logits / (T or TEMPERATURE), 1)
    top2 = p.topk(2, 1).values
    sure = (top2[:, 0] >= conf) & ((top2[:, 0] - top2[:, 1]) >= margin)
    return p.argmax(1), sure

def report(name, logits, y):
    pred, sure = decide(logits)
    acc_all = (pred == y).float().mean().item()
    cov = sure.float().mean().item()
    acc_sure = (pred[sure] == y[sure]).float().mean().item() if sure.any() else float("nan")
    print(f"{name}: precisión total {acc_all:.3f} | responde en {cov:.1%} de los casos | precisión cuando responde {acc_sure:.3f}")
    return {"n": int(len(y)), "accuracy_all": acc_all, "coverage": cov, "accuracy_when_answering": acc_sure, "escalation_rate": 1 - cov}

metrics = {"in_distribution_val": report("Validación (dentro de distribución)", val_logits, val_y)}

## 6. Prueba fuera de distribución (el número que va al video)

Uganda y Perú: fincas, cámaras y países que el modelo nunca vio. Las imágenes de clases que el modelo no conoce (por ejemplo, ojo de gallo en Perú) sirven para medir si la app sabe decir "no estoy segura".

In [ ]:
def plot_cm(name, y, pred):
    cm = confusion_matrix(y, pred, labels=list(range(len(CLASSES))))
    fig, ax = plt.subplots(figsize=(5, 4)); ax.imshow(cm, cmap="Greens")
    ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES, rotation=45, ha="right")
    ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(CLASSES)
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)): ax.text(j, i, cm[i, j], ha="center", va="center")
    ax.set_xlabel("Predicción"); ax.set_ylabel("Real"); ax.set_title(name)
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/confusion_{name}.png", dpi=120); plt.show()

plot_cm("validacion", val_y.numpy(), val_logits.argmax(1).numpy())

ood_cache = {}
for key in ["uganda", "peru"]:
    items = ext.get(key, [])
    known = [(p, l) for p, l in items if l in CLASSES]
    unknown = [(p, "sana") for p, l in items if l not in CLASSES]   # etiqueta ficticia; solo se mide la abstención
    if known:
        lg, yy = get_logits(model, DataLoader(FileSet(known, eval_tf), BATCH, num_workers=2))
        ood_cache[key] = (known, lg, yy)
        metrics[f"ood_{key}"] = report(f"{key} (fuera de distribución)", lg, yy)
        plot_cm(key, yy.numpy(), lg.argmax(1).numpy())
        present = sorted(set(yy.tolist()))
        print(classification_report(yy, lg.argmax(1), labels=present, target_names=[CLASSES[i] for i in present], zero_division=0))
    if unknown:
        lg, _ = get_logits(model, DataLoader(FileSet(unknown, eval_tf), BATCH, num_workers=2))
        _, sure = decide(lg)
        rate = 1 - sure.float().mean().item()
        metrics[f"ood_{key}_unknown_class_abstention"] = {"n": len(unknown), "abstention_rate": rate}
        print(f"{key}: ante {len(unknown)} imágenes de clases que no conoce, se abstiene en {rate:.1%} (ideal: cerca de 100%)")
if not ood_cache:
    print("No hay datos externos. Sin esta prueba no tienen un número honesto: revisa la sección 2.")

### Ajuste del umbral
La tabla muestra, para cada umbral de confianza, en qué porcentaje de casos responde la app y con qué precisión. Elijan el umbral con la prueba externa, no con la validación, y actualicen `CONF_THRESHOLD` en Configuración si lo cambian.

In [ ]:
src = None
if ood_cache:
    lg = torch.cat([v[1] for v in ood_cache.values()]); yy = torch.cat([v[2] for v in ood_cache.values()]); src = "externo"
else:
    lg, yy = val_logits, val_y; src = "validación"
print("Fuente:", src)
print("umbral | responde | precisión cuando responde")
for th in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    pred, sure = decide(lg, conf=th)
    a = (pred[sure] == yy[sure]).float().mean().item() if sure.any() else float("nan")
    print(f" {th:.2f}  |  {sure.float().mean().item():6.1%}  |  {a:.3f}")

## 7. Exportar a ONNX y cuantizar

In [ ]:
import onnx, onnxruntime as ort
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType, QuantFormat

model_cpu = model.to("cpu").eval()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
FP32 = f"{OUT_DIR}/model_fp32.onnx"; INT8 = f"{OUT_DIR}/model_int8.onnx"
kw = dict(input_names=["input"], output_names=["logits"], opset_version=17)
try:
    torch.onnx.export(model_cpu, dummy, FP32, dynamo=False, **kw)
except TypeError:
    torch.onnx.export(model_cpu, dummy, FP32, **kw)
onnx.checker.check_model(onnx.load(FP32))

class Calib(CalibrationDataReader):
    def __init__(self, n=200):
        sub = random.sample(train_idx, n)
        self.it = iter([{"input": HFSet([i], eval_tf)[0][0].unsqueeze(0).numpy()} for i in sub])
    def get_next(self): return next(self.it, None)

int8_ok = True
try:
    pre = f"{OUT_DIR}/_pre.onnx"
    try:
        from onnxruntime.quantization.shape_inference import quant_pre_process
        quant_pre_process(FP32, pre)
    except Exception:
        shutil.copy(FP32, pre)
    quantize_static(pre, INT8, Calib(), quant_format=QuantFormat.QDQ,
                    activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8)
    os.remove(pre)
except Exception as e:
    int8_ok = False; print("La cuantización falló:", e)

def onnx_logits(path, dataset, n=1500):
    sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    idx = random.sample(range(len(dataset)), min(n, len(dataset)))
    L, Y, t = [], [], 0
    for i in idx:
        x, y = dataset[i]; t0 = time.time()
        L.append(sess.run(None, {"input": x.unsqueeze(0).numpy()})[0][0]); t += time.time() - t0; Y.append(y)
    return torch.tensor(np.array(L)), torch.tensor(Y), 1000 * t / len(idx)

sizes = {}
for tag, path in [("fp32", FP32), ("int8", INT8)]:
    if tag == "int8" and not int8_ok: continue
    sizes[tag] = round(os.path.getsize(path) / 1e6, 2)
    lg, yy, ms = onnx_logits(path, val_ds)
    metrics[f"onnx_{tag}_val"] = report(f"ONNX {tag} validación ({sizes[tag]} MB, {ms:.0f} ms en CPU de Colab)", lg, yy)
    for key, (known, _, _) in ood_cache.items():
        lg, yy, _ = onnx_logits(path, FileSet(known, eval_tf))
        metrics[f"onnx_{tag}_ood_{key}"] = report(f"ONNX {tag} {key}", lg, yy)

recommended = "fp32"
if int8_ok:
    ref = [k for k in metrics if k.startswith("onnx_fp32_ood_")] or ["onnx_fp32_val"]
    drop = max(metrics[k]["accuracy_all"] - metrics[k.replace("fp32", "int8")]["accuracy_all"] for k in ref)
    recommended = "int8" if drop <= 0.02 else "fp32"
    print(f"\nCaída máxima por cuantización: {drop:.3f} → usar {recommended}")
if recommended == "fp32" and sizes.get("fp32", 0) > 10:
    print("El modelo sin cuantizar supera 10 MB. Ejecuta de nuevo con MODEL_NAME = 'mobilenetv3_small_100' (plan B).")

## 8. Ficha del modelo y descarga

In [ ]:
card = {
    "id": f"arabica-v1-{MODEL_NAME}",
    "architecture": MODEL_NAME,
    "classes": CLASSES,
    "input": {"name": "input", "shape": [1, 3, IMG_SIZE, IMG_SIZE], "layout": "NCHW", "color": "RGB",
              "resize": "estirar a 224x224 (sin recorte)", "scale": "dividir entre 255", "mean": MEAN, "std": STD},
    "output": {"name": "logits", "postprocess": "softmax(logits / temperature)"},
    "temperature": TEMPERATURE,
    "unsure_rule": {"min_confidence": CONF_THRESHOLD, "min_margin": MARGIN_THRESHOLD},
    "files": {"fp32": "model_fp32.onnx", "int8": "model_int8.onnx" if int8_ok else None},
    "recommended_file": "model_int8.onnx" if recommended == "int8" else "model_fp32.onnx",
    "size_mb": sizes,
    "metrics": metrics,
    "training_data": [
        {"name": "JMuBEN / JMuBEN2", "country": "Kenia", "license": "CC BY 4.0",
         "source": "https://huggingface.co/datasets/Project-AgML/arabica_coffee_leaf_disease_classification",
         "doi": ["10.17632/t2r6rszp5c.1", "10.17632/tgv3zb82nd.1"], "images_used": len(train_idx)},
        {"name": "BRACOL", "country": "Brasil", "license": "confirmar en la página",
         "source": "https://data.mendeley.com/datasets/yy2k5y8mxg/1", "images_used": len(bracol_known)},
    ],
    "evaluation_data_never_trained_on": [
        {"name": "Uganda (Soroti University)", "source": "https://data.mendeley.com/datasets/k36wnd6knb/1"},
        {"name": "Perú (Saposoa)", "source": "https://data.mendeley.com/datasets/mfpxg4y65r/1"},
    ],
    "known_limits": [
        "Solo hojas de arábica; no cubre frutos ni robusta.",
        "JMuBEN son hojas recortadas y con aumentación: la validación interna está inflada.",
        "No reconoce deficiencias nutricionales ni enfermedades fuera de las 5 clases.",
    ],
}
json.dump(card, open(f"{OUT_DIR}/model_card.json", "w"), indent=2, ensure_ascii=False)
if os.path.exists(f"{OUT_DIR}/best.pt"): os.remove(f"{OUT_DIR}/best.pt")
shutil.make_archive(OUT_DIR, "zip", OUT_DIR)
print(json.dumps(card["metrics"], indent=2))
try:
    from google.colab import files; files.download(f"{OUT_DIR}.zip")
except Exception:
    print("Descarga manual:", f"{OUT_DIR}.zip")